# Production Agent Validation: Reducing Hallucinations with Amazon Bedrock AgentCore

Test the hotel booking agent locally against DynamoDB before deploying to Amazon Bedrock AgentCore. This notebook validates steering rules, payment enforcement, and error handling — the same anti-hallucination techniques from demos 01-05 applied in a production architecture.

- **Part 1** — Agent WITHOUT validation (baseline: shows how rules get bypassed)
- **Part 2** — Agent WITH `validate_booking_rules` (symbolic rules catch violations)
- **Part 3** — Agent deployed on AgentCore Runtime (production behavior)

**Prerequisites** (requires an AWS account and OpenAI API access, which may have associated costs — see [AWS Free Tier](https://aws.amazon.com/free/) and [OpenAI pricing](https://openai.com/pricing) for details):
1. Deploy the infrastructure using [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) (a tool for defining cloud infrastructure as code): `cdk deploy HotelBookingAgentStack`
2. Load sample hotel data into DynamoDB: `uv run python seed_data.py`
3. Set `OPENAI_API_KEY` in your environment ([get one here](https://platform.openai.com/api-keys))
4. Configure [AWS credentials](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-quickstart.html) with appropriate permissions

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) — invocation logs, tool call traces, and error tracking — with no additional setup.

# Hotel Booking Agent — Testing

Test the booking agent locally and on [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Runtime.

- **Part 1** — Agent WITHOUT validation (baseline: shows how rules get bypassed)
- **Part 2** — Agent WITH `validate_booking_rules` (symbolic rules catch violations)
- **Part 3** — Agent deployed on AgentCore Runtime (production behavior)

**Prerequisites** (requires an AWS account and OpenAI API access, which may have associated costs — see [AWS Free Tier](https://aws.amazon.com/free/) and [OpenAI pricing](https://openai.com/pricing) for details):
1. Deploy the infrastructure using [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/getting_started.html) (a tool for defining cloud infrastructure as code): `cdk deploy HotelBookingAgentStack`
2. Load sample hotel data into DynamoDB: `uv run python seed_data.py`
3. Set `OPENAI_API_KEY` in your environment ([get one here](https://platform.openai.com/api-keys))
4. Configure [AWS credentials](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-quickstart.html) with appropriate permissions

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html) — invocation logs, tool call traces, and error tracking — with no additional setup.

In [ ]:
import os
import warnings

os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
warnings.filterwarnings("ignore", message="Failed to detach context")


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys "
        "and export it: export OPENAI_API_KEY=sk-..."
    )

In [3]:
# OpenAI-compatible model interface
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id="gpt-4o-mini")

In [4]:
from local_tools import ALL_TOOLS

print(f"Loaded {len(ALL_TOOLS)} tools:")
for t in ALL_TOOLS:
    print(f"  - {t.__name__}")

Loaded 7 tools:
  - search_available_hotels
  - book_hotel
  - get_booking
  - process_payment
  - confirm_booking
  - cancel_booking
  - validate_booking_rules


## Part 1: Agent WITHOUT validation tool

Test with only the operational tools (CRUD) — without `validate_booking_rules`.
This shows how the agent can bypass business rules.

In [5]:
CRUD_TOOLS = [t for t in ALL_TOOLS if t.__name__ != "validate_booking_rules"]

SYSTEM_PROMPT = (
    "You are a hotel booking assistant. Help users search, book, pay, "
    "confirm, and cancel hotel reservations. Use the tools provided."
)

agent_no_validation = Agent(
    model=MODEL,
    tools=CRUD_TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

### Test 1a: Happy path (should work)

In [6]:
agent_no_validation("Search for hotels in Paris with at least 4 stars")


Tool #1: search_available_hotels


I found one available hotel in Paris with at least 4 stars:



- **Grand Hotel Paris**
  - Stars: 5


  - Price: $350/night
 

 - Rooms available: 12
  - Hotel

...

 like to book a room at this hotel? If so,

 please provide the following details:
1. Guest name
2. Check-in date

 (YYYY-MM-DD)
3. Check-out date (YYYY-MM-DD)
4

. Number of guests (optional, default is 1)

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I found one available hotel in Paris with at least 4 stars:\n\n- **Grand Hotel Paris**\n  - Stars: 5\n  - Price: $350/night\n  - Rooms available: 12\n  - Hotel ID: grand-hotel-paris\n\nWould you like to book a room at this hotel? If so, please provide the following details:\n1. Guest name\n2. Check-in date (YYYY-MM-DD)\n3. Check-out date (YYYY-MM-DD)\n4. Number of guests (optional, default is 1)'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'search_available_hotels': ToolMetrics(tool={'toolUseId': 'call_Q9yBmfr1bCuouvuyZZstnXvD', 'name': 'search_available_hotels', 'input': {'city': 'Paris', 'min_stars': 4}}, call_count=1, success_count=1, error_count=0, total_time=0.2930171489715576)}, cycle_durations=[2.374569892883301], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='ffba4e08-eaa5-4820-8f08-61872e24a477', usage={'inputTokens': 538, 'outputTokens': 

### Test 1b: 15 guests — should fail, but agent may proceed without checking

In [7]:
agent_no_validation(
    "Book the Budget Inn Barcelona for Alex for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03"
)

It seems that "Budget Inn Barcelona" was not part of the search results for

 hotels in Paris.

The only available hotel in Paris is "Grand Hotel Paris."

 Would you like to book a room there for Alex instead? Please provide the required

 details:

1. Guest name (Alex)
2. Check-in date (202

6-06-01)
3.

 Check-out date (2026-06-03)
4. Number

 of guests (15) 

Let me know if you want to proceed

 with this hotel or if you want to search for hotels in

 Barcelona instead.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'It seems that "Budget Inn Barcelona" was not part of the search results for hotels in Paris.\n\nThe only available hotel in Paris is "Grand Hotel Paris." Would you like to book a room there for Alex instead? Please provide the required details:\n\n1. Guest name (Alex)\n2. Check-in date (2026-06-01)\n3. Check-out date (2026-06-03)\n4. Number of guests (15) \n\nLet me know if you want to proceed with this hotel or if you want to search for hotels in Barcelona instead.'}]}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'search_available_hotels': ToolMetrics(tool={'toolUseId': 'call_Q9yBmfr1bCuouvuyZZstnXvD', 'name': 'search_available_hotels', 'input': {'city': 'Paris', 'min_stars': 4}}, call_count=1, success_count=1, error_count=0, total_time=0.2930171489715576)}, cycle_durations=[2.374569892883301, 2.2808620929718018], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_

### Test 1c: Confirm without payment — agent has no rule to prevent this

In [8]:
agent_no_validation(
    "Book the Berlin Central Hotel for Jordan, 2 guests, "
    "check-in 2026-07-01, check-out 2026-07-05. "
    "Then immediately confirm the booking without paying."
)

It appears that "Berlin Central Hotel" was not part of the search

 results for hotels, as I have only found the "Grand Hotel Paris" in

 Paris with at least 4 stars.

Would you like me to search

 for hotels in Berlin instead? If so, please let me know the criteria you

 would like to use for the search (such as star rating

 or maximum price). If you want to go ahead with "Grand Hotel

 Paris" or any specific hotel in Berlin, please provide the details.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'It appears that "Berlin Central Hotel" was not part of the search results for hotels, as I have only found the "Grand Hotel Paris" in Paris with at least 4 stars.\n\nWould you like me to search for hotels in Berlin instead? If so, please let me know the criteria you would like to use for the search (such as star rating or maximum price). If you want to go ahead with "Grand Hotel Paris" or any specific hotel in Berlin, please provide the details.'}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'search_available_hotels': ToolMetrics(tool={'toolUseId': 'call_Q9yBmfr1bCuouvuyZZstnXvD', 'name': 'search_available_hotels', 'input': {'city': 'Paris', 'min_stars': 4}}, call_count=1, success_count=1, error_count=0, total_time=0.2930171489715576)}, cycle_durations=[2.374569892883301, 2.2808620929718018, 2.0196118354797363], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_c

## Part 2: Agent WITH validation tool

Now add `validate_booking_rules` and instruct the agent to always validate first.
The symbolic rules catch violations that the LLM would otherwise miss.

In [9]:
GUARDED_PROMPT = (
    "You are a hotel booking assistant. Help users search, book, pay, "
    "confirm, and cancel hotel reservations.\n\n"
    "IMPORTANT RULES:\n"
    "- ALWAYS call validate_booking_rules BEFORE calling book_hotel, "
    "confirm_booking, or cancel_booking.\n"
    "- If validation returns FAIL, inform the user about the violation "
    "and do NOT proceed with the action.\n"
    "- Follow the booking flow: search -> validate -> book -> pay -> validate -> confirm."
)

agent_with_validation = Agent(
    model=MODEL,
    tools=ALL_TOOLS,
    system_prompt=GUARDED_PROMPT,
)

### Test 2a: Happy path — full booking flow

In [10]:
agent_with_validation(
    "Book the Grand Hotel Paris for Sam, 2 guests, "
    "check-in 2026-06-10, check-out 2026-06-13. "
    "Then pay the full amount and confirm."
)


Tool #1: validate_booking_rules



Tool #2: book_hotel



Tool #3: process_payment



Tool #4: validate_booking_rules



Tool #5: confirm_booking


...

:** 2026-06-13
- **Number of Guests

:** 2
- **Total Amount Paid:** $105

0
- **Status:** CONFIRMED

If you

 need any further assistance, feel free to ask!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Your booking at the Grand Hotel Paris has been successfully made for Sam. Here are the details:\n\n- **Booking ID:** BK-39C1B30C\n- **Hotel:** Grand Hotel Paris\n- **Check-in Date:** 2026-06-10\n- **Check-out Date:** 2026-06-13\n- **Number of Guests:** 2\n- **Total Amount Paid:** $1050\n- **Status:** CONFIRMED\n\nIf you need any further assistance, feel free to ask!'}]}, metrics=EventLoopMetrics(cycle_count=6, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_DmUQJSH6pOzwRHpCRjINyLGL', 'name': 'validate_booking_rules', 'input': {'action': 'confirm', 'booking_id': 'BK-39C1B30C'}}, call_count=2, success_count=2, error_count=0, total_time=0.24892663955688477), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_ujpRxWlzMYfJsShhoX2mfxlY', 'name': 'book_hotel', 'input': {'hotel_id': 'grand-hotel-paris', 'guest_name': 'Sam', 'check_in': '2026-06-10', 'check_out': '2026-06-13', 'g

### Test 2b: 15 guests — validation catches the violation

In [11]:
agent_with_validation(
    "Book the Budget Inn Barcelona for Alex for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03"
)


Tool #6: validate_booking_rules


The maximum number of guests for a booking is 10, but you requested a

 reservation for 15 guests. I can adjust your reservation to

 10 guests. Would you like to proceed with that, or would

 you prefer to make separate bookings for the remaining guests?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The maximum number of guests for a booking is 10, but you requested a reservation for 15 guests. I can adjust your reservation to 10 guests. Would you like to proceed with that, or would you prefer to make separate bookings for the remaining guests?'}]}, metrics=EventLoopMetrics(cycle_count=8, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_ylUzlYzJHP8WdF1LtF8NHdMe', 'name': 'validate_booking_rules', 'input': {'action': 'book', 'guests': 15, 'check_in': '2026-06-01', 'check_out': '2026-06-03'}}, call_count=3, success_count=3, error_count=0, total_time=0.331850528717041), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_ujpRxWlzMYfJsShhoX2mfxlY', 'name': 'book_hotel', 'input': {'hotel_id': 'grand-hotel-paris', 'guest_name': 'Sam', 'check_in': '2026-06-10', 'check_out': '2026-06-13', 'guests': 2}}, call_count=1, success_count=1, error_count=0, total_time=0.26335000991821

### Test 2c: Confirm without payment — validation blocks it

In [12]:
agent_with_validation(
    "Book the Tokyo Tower Hotel for Riley, 1 guest, "
    "check-in 2026-08-01, check-out 2026-08-03. "
    "Skip payment and confirm directly."
)


Tool #7: validate_booking_rules



Tool #8: book_hotel



Tool #9: validate_booking_rules


I need to process your payment of $560 before confirming your booking. Shall I

 proceed with the payment?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'I need to process your payment of $560 before confirming your booking. Shall I proceed with the payment?'}]}, metrics=EventLoopMetrics(cycle_count=12, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_4KBLdIIUEm8CoNhNtvG2zphM', 'name': 'validate_booking_rules', 'input': {'action': 'confirm', 'booking_id': 'BK-7AF13C51'}}, call_count=5, success_count=5, error_count=0, total_time=0.5736503601074219), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_yfajEWD6owY9pq0Hzy6LspIH', 'name': 'book_hotel', 'input': {'hotel_id': 'tokyo-tower-hotel', 'guest_name': 'Riley', 'check_in': '2026-08-01', 'check_out': '2026-08-03', 'guests': 1}}, call_count=2, success_count=2, error_count=0, total_time=0.506005048751831), 'process_payment': ToolMetrics(tool={'toolUseId': 'call_QYtxAMJmFxzQqaDYUYOc3pm2', 'name': 'process_payment', 'input': {'booking_id': 'BK-39C1B30C', 'amount': 1050}}, call_

### Test 2d: Hotel with no availability

In [13]:
agent_with_validation(
    "Book the Roma Classic Hotel for Taylor, 1 guest, "
    "check-in 2026-06-15, check-out 2026-06-17"
)


Tool #10: validate_booking_rules



Tool #11: book_hotel


Unfortunately, there are no available

 rooms at the Roma Classic Hotel for your requested dates (check-in:

 2026-06-15, check-out: 2026

-06-17). Would you like me to search for available hotels in the area?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Unfortunately, there are no available rooms at the Roma Classic Hotel for your requested dates (check-in: 2026-06-15, check-out: 2026-06-17). Would you like me to search for available hotels in the area?'}]}, metrics=EventLoopMetrics(cycle_count=15, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_IRrPIyIpOAqftbyoqMrliQcb', 'name': 'validate_booking_rules', 'input': {'action': 'book', 'guests': 1, 'check_in': '2026-06-15', 'check_out': '2026-06-17'}}, call_count=6, success_count=6, error_count=0, total_time=0.6582603454589844), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_SMbtemx9Ep1rYU2NAIdplqLx', 'name': 'book_hotel', 'input': {'hotel_id': 'roma-classic-hotel', 'guest_name': 'Taylor', 'check_in': '2026-06-15', 'check_out': '2026-06-17', 'guests': 1}}, call_count=3, success_count=3, error_count=0, total_time=0.5850911140441895), 'process_payment': ToolMetrics(tool=

### Test 2e: Search in a city with no hotels

In [14]:
agent_with_validation("Find available hotels in Reykjavik")


Tool #12: search_available_hotels


I'm sorry, but there are currently no available hotels in Reykjavik.

 If you're flexible with your dates or destination, please let me know, and I

 can assist you further!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I'm sorry, but there are currently no available hotels in Reykjavik. If you're flexible with your dates or destination, please let me know, and I can assist you further!"}]}, metrics=EventLoopMetrics(cycle_count=17, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_IRrPIyIpOAqftbyoqMrliQcb', 'name': 'validate_booking_rules', 'input': {'action': 'book', 'guests': 1, 'check_in': '2026-06-15', 'check_out': '2026-06-17'}}, call_count=6, success_count=6, error_count=0, total_time=0.6582603454589844), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_SMbtemx9Ep1rYU2NAIdplqLx', 'name': 'book_hotel', 'input': {'hotel_id': 'roma-classic-hotel', 'guest_name': 'Taylor', 'check_in': '2026-06-15', 'check_out': '2026-06-17', 'guests': 1}}, call_count=3, success_count=3, error_count=0, total_time=0.5850911140441895), 'process_payment': ToolMetrics(tool={'toolUseId': 'call_QYtxAMJmFxzQqa

### Test 2f: Book a non-existent hotel — agent should not hallucinate

In [15]:
agent_with_validation(
    "Book the Ritz Carlton Singapore for Morgan, 2 guests, "
    "check-in 2026-06-20, check-out 2026-06-22"
)


Tool #13: validate_booking_rules



Tool #14: book_hotel


It seems that I don't have

 the Ritz Carlton Singapore in my

 records. Would you

 like me to search

 for available hotels in

 Singapore instead?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "It seems that I don't have the Ritz Carlton Singapore in my records. Would you like me to search for available hotels in Singapore instead?"}]}, metrics=EventLoopMetrics(cycle_count=20, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_uV5ncC5WBVeYZtd4AwQ0i6Ue', 'name': 'validate_booking_rules', 'input': {'action': 'book', 'guests': 2, 'check_in': '2026-06-20', 'check_out': '2026-06-22'}}, call_count=7, success_count=7, error_count=0, total_time=0.7400493621826172), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_BhxMX9tIZq9y8z2xfDGCMHNV', 'name': 'book_hotel', 'input': {'hotel_id': 'ritz-carlton-singapore', 'guest_name': 'Morgan', 'check_in': '2026-06-20', 'check_out': '2026-06-22', 'guests': 2}}, call_count=4, success_count=4, error_count=0, total_time=0.6629829406738281), 'process_payment': ToolMetrics(tool={'toolUseId': 'call_QYtxAMJmFxzQqaDYUYOc3pm2', 'name': 'proc

### Test 2g: Retrieve a non-existent booking

In [16]:
agent_with_validation("Check the status of booking BK-99999999")


Tool #15: get_booking


There is no booking associated with the ID 'BK

-99999999'. Please double-check

 the booking ID and try again, or let me know if you need help with anything else!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "There is no booking associated with the ID 'BK-99999999'. Please double-check the booking ID and try again, or let me know if you need help with anything else!"}]}, metrics=EventLoopMetrics(cycle_count=22, tool_metrics={'validate_booking_rules': ToolMetrics(tool={'toolUseId': 'call_uV5ncC5WBVeYZtd4AwQ0i6Ue', 'name': 'validate_booking_rules', 'input': {'action': 'book', 'guests': 2, 'check_in': '2026-06-20', 'check_out': '2026-06-22'}}, call_count=7, success_count=7, error_count=0, total_time=0.7400493621826172), 'book_hotel': ToolMetrics(tool={'toolUseId': 'call_BhxMX9tIZq9y8z2xfDGCMHNV', 'name': 'book_hotel', 'input': {'hotel_id': 'ritz-carlton-singapore', 'guest_name': 'Morgan', 'check_in': '2026-06-20', 'check_out': '2026-06-22', 'guests': 2}}, call_count=4, success_count=4, error_count=0, total_time=0.6629829406738281), 'process_payment': ToolMetrics(tool={'toolUseId': 'call_QYtxAMJmFxzQqaDYUYOc

## Part 3: Test the deployed agent on AgentCore Runtime

Run the same scenarios against the agent deployed on [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/).
This confirms that the production agent behaves identically to the local version.

AgentCore provides [built-in observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)
— invocation logs, tool call traces, and error tracking — with no additional configuration.

In [ ]:
import boto3
import json
import base64
import subprocess

# Get ARN dynamically from CloudFormation outputs
cf = boto3.client("cloudformation", region_name="us-east-1")
stack_outputs = cf.describe_stacks(StackName="HotelBookingAgentStack")["Stacks"][0]["Outputs"]
AGENT_RUNTIME_ARN = next(o["OutputValue"] for o in stack_outputs if o["OutputKey"] == "AgentRuntimeArn")

def ask_agent(prompt: str) -> str:
    """Send a prompt to the deployed AgentCore agent and return the response."""
    payload = base64.b64encode(json.dumps({"prompt": prompt}).encode()).decode()
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
        outfile = f.name
    result = subprocess.run(
        ["aws", "bedrock-agentcore", "invoke-agent-runtime",
         "--agent-runtime-arn", AGENT_RUNTIME_ARN,
         "--payload", payload,
         "--region", "us-east-1",
         outfile],
        capture_output=True, text=True, timeout=120
    )
    with open(outfile) as f:
        response = f.read()
    os.unlink(outfile)
    try:
        return json.loads(response)
    except (json.JSONDecodeError, TypeError):
        return response

print(f"Connected to AgentCore Runtime")


### Test 3a: Search hotels (AgentCore)

In [18]:
print(ask_agent("Search for available hotels in Paris"))

I found 1 available hotel in Paris:

- **Grand Hotel Paris**: 5 stars, $350 per night, 11 rooms available | **ID**: grand-hotel-paris

Would you like to proceed with booking this hotel? If so, please provide your check-in and check-out dates, as well as the number of guests.



### Test 3b: Full booking flow (AgentCore)

In [19]:
print(ask_agent(
    "Book the Grand Hotel Paris for Sam, 2 guests, "
    "check-in 2026-06-10, check-out 2026-06-13. "
    "Then pay and confirm the booking."
))

Your booking for the Grand Hotel Paris has been successfully made for Sam, with 2 guests, from June 10, 2026, to June 13, 2026. 

Here are the details:
- **Booking ID:** BK-CEDF44E9
- **Total Amount:** $1050
- **Booking Status:** CONFIRMED

If you need any further assistance, feel free to ask!



### Test 3c: Soft steering — 15 guests (AgentCore)

In [20]:
print(ask_agent(
    "Book Budget Inn Barcelona for 15 guests, "
    "check-in 2026-06-01, check-out 2026-06-03."
))

Booking for 15 guests is not available, but you can book for up to 10 guests. I adjusted your reservation to 10 guests (our maximum). Would you like to proceed with the booking or make separate bookings for the remaining guests?



### Test 3d: Non-existent hotel (AgentCore)

In [21]:
print(ask_agent(
    "Book the Ritz Carlton Singapore for 2 guests, "
    "check-in 2026-06-20, check-out 2026-06-22."
))

It seems that I couldn't find the Ritz Carlton Singapore available for your requested dates from June 20 to June 22, 2026. It appears that there are no hotels available in Singapore during this time.

Would you like me to help you with alternative dates or search for other hotels in Singapore?



## Results Summary

### Part 1 vs Part 2: Local agent

| # | Scenario | Without validation | With validation |
|---|----------|-------------------|------------------|
| a | Search hotels | Works | Works |
| b | 15 guests (max 10) | May proceed or rely on tool error | FAIL before calling book_hotel |
| c | Confirm without payment | May attempt confirmation | FAIL: status must be PAID |
| d | Sold-out hotel | Tool returns error | Tool returns error |
| e | City with no hotels | Returns empty | Returns empty (no hallucination) |
| f | Non-existent hotel | May hallucinate | Tool returns error |
| g | Non-existent booking | May hallucinate | Tool returns error |

### Part 3: AgentCore Runtime

| # | Scenario | Anti-hallucination layer | Expected behavior |
|---|----------|------------------------|-----------------------|
| a | Search hotels | Grounded retrieval | Returns real DynamoDB data |
| b | Full booking flow | All layers active | validate -> book -> pay -> validate -> confirm |
| c | 15 guests (max 10) | Soft steering (DynamoDB rules) | Agent self-corrects to 10 guests |
| d | Non-existent hotel | Grounded retrieval | Tool returns error, no hallucination |

The `validate_booking_rules` tool adds a symbolic safety layer that the LLM
cannot bypass — business rules are enforced as code, not as prompt suggestions.